In [2]:
import os
import pdfplumber

In [3]:

from pathlib import Path

PATH = "data/zbior_danych"

def get_pdf_paths(PATH, as_str: bool = False):
    """
        INPUT: PATH - folder startowy; as_str - czy zwrócić ścieżki jako stringi.
        OUTPUT: Lista ścieżek do plików PDF.
        DESCRIPTION: Rekurencyjnie wyszukuje wszystkie pliki .pdf w podanym katalogu.
    """
    dir = Path(PATH)
    
    # rglob("*.pdf") znajdzie pliki .pdf we wszystkich podfolderach
    pliki_pdf = dir.rglob("*.pdf")
    
    if as_str:
        return [str(plik) for plik in pliki_pdf]
    else:
        return list(pliki_pdf)

In [4]:
pdf_paths = get_pdf_paths(PATH, as_str=True)
pdf_paths[:3]

['data\\zbior_danych\\www.sn.pl\\2026-01\\iii szp 2-05_38097a23.pdf',
 'data\\zbior_danych\\www.sn.pl\\2026-01\\Informacja dodatkowa SÄ…du NajwyÅ¼szego za 2020 r_a7df981f.pdf',
 'data\\zbior_danych\\www.sn.pl\\2026-01\\Informacja o wynikach kontroli wykonania budÅ¼etu  paÅ„stwa w 2021 roku w czÄ™Å›ci 04   _05e59cbc.pdf']

In [4]:
print(len(pdf_paths))

108


In [6]:
def extract_text_from_pdf(pdf_path):
    """
        INPUT: pdf_path - ścieżka do pojedynczego pliku PDF.
        OUTPUT: Tekst wyciągnięty z PDF-a jako jeden string.
        DESCRIPTION: Odczytuje strony PDF-a przez pdfplumber, łączy tekst i normalizuje białe znaki.
    """
    pdf_text = ""

    print(f"Text extraction from: {os.path.basename(pdf_path)}\n")
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            temp_text = page.extract_text(layout=True)
            pdf_text += temp_text + "\n"
            pdf_text = " ".join(pdf_text.split())
    print(f"Extracted text from {os.path.basename(pdf_path)}: {len(pdf_text)} characters\n")

    
    return pdf_text

In [7]:
pdf_text = extract_text_from_pdf(pdf_paths[0])
pdf_text[:100]

Text extraction from: iii szp 2-05_38097a23.pdf

Extracted text from iii szp 2-05_38097a23.pdf: 52519 characters



'Wyrok z dnia 20 września 2005 r. III SZP 2/05 1. Jurysdykcyjna funkcja Sądu Okręgowego-Sądu Ochrony '

In [8]:
import re

def tokenize_text(pdf_text):
    """
        INPUT: pdf_text - pełny tekst dokumentu PDF.
        OUTPUT: Lista tokenów z polami token, start, end i label.
        DESCRIPTION: Dzieli tekst na tokeny i nadaje każdemu domyślną etykietę O.
    """
    token_pattern = re.compile(
    r"\d{2}-\d{3}|[\w]+|[^\w\s]",
    flags=re.UNICODE
    )
    pdf_tokens = []

    for match in token_pattern.finditer(pdf_text):
        pdf_tokens.append({
            "token": match.group(),
            "start": match.start(),
            "end": match.end(),
            "label": 'O'
        })

    return pdf_tokens

In [9]:
pdf_tokens = tokenize_text(pdf_text)
pdf_tokens[:3]

[{'token': 'Wyrok', 'start': 0, 'end': 5, 'label': 'O'},
 {'token': 'z', 'start': 6, 'end': 7, 'label': 'O'},
 {'token': 'dnia', 'start': 8, 'end': 12, 'label': 'O'}]

In [10]:
def create_chunks(
    text: str,
    tokens: list[dict]) -> list[dict]:

    """
        INPUT: text - pełny tekst dokumentu; tokens - lista tokenów z pozycjami znakowymi.
        OUTPUT: Lista chunków zawierających text, start i end.
        DESCRIPTION: Dzieli dokument na nakładające się fragmenty po 180 tokenów z overlapem 30.
    """
    
    CHUNK_SIZE = 180
    OVERLAP = 30
    
    if not tokens:
        return []

    step = CHUNK_SIZE - OVERLAP
    chunks = []

    for token_start_index in range(0, len(tokens), step):
        token_end_index = min(
            token_start_index + CHUNK_SIZE,
            len(tokens)
        )

        chunk_tokens = tokens[token_start_index:token_end_index]

        if not chunk_tokens:
            break

        chunk_start = chunk_tokens[0]["start"]
        chunk_end = chunk_tokens[-1]["end"]

        chunks.append({
            "text": text[chunk_start:chunk_end],
            "start": chunk_start,
            "end": chunk_end
        })

        if token_end_index >= len(tokens):
            break

    return chunks

In [11]:

chunks = create_chunks(
    text=pdf_text,
    tokens=pdf_tokens)

print(chunks[0])

{'text': 'Wyrok z dnia 20 września 2005 r. III SZP 2/05 1. Jurysdykcyjna funkcja Sądu Okręgowego-Sądu Ochrony Konkurencji i Konsumentów nie może sprowadzać się tylko do oceny legalności decyzji Pre- zesa Urzędu Ochrony Konkurencji i Konsumentów. Sąd ten powinien dążyć do ustalenia okoliczności faktycznych sprawy, a następnie dokonać ich prawnej oceny w zakresie zasadności odwołania. 2. Przedmiotem zagadnienia prawnego przedstawionego Sądowi Najwyż- szemu (art. 390 § 1 k.p.c.) może być wyłącznie kwestia budząca poważne wąt- pliwości prawne, której rozstrzygnięcie jest niezbędne do rozpoznania apelacji. Samoistnej przesłanki wystąpienia z pytaniem prawnym nie stanowi natomiast waga problemu ani rozbieżności w orzecznictwie lub piśmiennictwie. Przewodniczący SSN Katarzyna Gonera (sprawozdawca), Sędziowie SN: Jerzy Kwaśniewski, Zbigniew Myszka. Sąd Najwyższy, po rozpoznaniu na rozprawie w dniu 20 września 2005 r. sprawy z odwołania „P.” SA w W. przeciwko Prezesowi Urzędu Regulacji Telekomu

In [16]:
from transformers import pipeline

pipe = pipeline("token-classification", model="lexedit/herbert-polish-legal-ner", aggregation_strategy="simple")

def classify_tokens(chunk: dict) -> list[dict]:
    """
        INPUT: chunk - fragment tekstu z polami text, start i end.
        OUTPUT: Lista encji wykrytych przez model NER.
        DESCRIPTION: Uruchamia globalny pipeline NER na tekście chunka i przelicza pozycje encji na pozycje w całym dokumencie.
    """
    
    results = pipe(chunk['text'])
    for result in results:
        result['start'] += chunk['start']
        result['end'] += chunk['start']
    return results

def filter_entities(entities):
    """
        INPUT: entities - lista encji zwróconych przez model.
        OUTPUT: Lista encji po odfiltrowaniu i zmianie nazw etykiet.
        DESCRIPTION: Usuwa niepotrzebne typy encji i mapuje etykiety modelu na PERSON, ORGANIZATION, CITY oraz STREET.
    """
    LABEL_MAPPING = {
        "PER": "PERSON",
        "ORG": "ORGANIZATION",
        "LOC_PUB": "CITY",
        "LOC": "STREET"
    }
    for entity in entities.copy():
        if entity['entity_group'] not in LABEL_MAPPING:
            entities.remove(entity)
        else:
            entity['entity_group'] = LABEL_MAPPING[entity['entity_group']]
    
    return entities


def validate_entities(entities, tokens):
    """
        INPUT: entities - lista encji z modelu; tokens - tokeny całego dokumentu.
        OUTPUT: Krotka: poprawne encje oraz encje odrzucone jako błędne.
        DESCRIPTION: Sprawdza, czy początek i koniec encji zgadzają się z tokenami dokumentu.
    """

    errors = []
    # Walidacja: sprawdzenie, czy tokeny w wynikach klasyfikacji zgadzają się z tokenami w całym tekście -> czy w pdfs_tokens[] na danej pozycji tokeny się zgadzają
    for entity in entities.copy():
        entity_length = len(entity['word'].split(' '))
        token_text_start = entity['word'].split(' ')[0]
        token_text_end = entity['word'].split(' ')[-1] if entity_length > 1 else token_text_start
        token_start = entity['start']
        token_end = entity['end']

        # Znalezienie tokenu w pdfs_tokens, który odpowiada wynikowi klasyfikacji
        if  entity_length > 1:
            starting_matching_tokens = [token for token in tokens if token['start'] == token_start and token['token'] == token_text_start]
            ending_matching_tokens = [token for token in tokens if token['end'] == token_end and token['token'] == token_text_end]
        else:
            starting_matching_tokens = [token for token in tokens if token['start'] == token_start and token['end'] == token_end and token['token'] == token_text_start]
            ending_matching_tokens = []
        
        if not starting_matching_tokens and not ending_matching_tokens:
            #print(f"Token '{token_text_start}' on pos '{token_start}' to '{token_end}' not found!")
            errors.append(entity)
            entities.remove(entity)
            
    return entities, errors

def flag_low_confidence(entities):
    """
        INPUT: entities - lista wykrytych encji.
        OUTPUT: Lista encji z confidence score poniżej 0.8.
        DESCRIPTION: Wyszukuje encje o niskiej pewności modelu, żeby można było je później ręcznie sprawdzić.
    """
    low_confidence_entities = []
    for entity in entities:
        if entity['score'] < 0.8:
            low_confidence_entities.append(entity)
    return low_confidence_entities


def mapping_entities_on_tokens(entities, tokens):
    """
        INPUT: entities - poprawne encje NER; tokens - tokeny dokumentu.
        OUTPUT: Lista tokenów z uzupełnionymi etykietami BIO.
        DESCRIPTION: Przenosi encje z poziomu znaków na konkretne tokeny i ustawia etykiety B- oraz I-.
    """
    continuation = False
    for token in tokens:
        #print(token['token'] + entities[0]['word'].split(' ')[0])
        if token['token'] == entities[0]['word'].split(' ')[0] and (token['start'] == entities[0]['start'] if continuation == False else True):
            token['label'] = 'B-' + entities[0]['entity_group'] if continuation == False else 'I-' + entities[0]['entity_group']
            total_entity_length = len(entities[0]['word'].split(' '))
            if total_entity_length > 1:
                entities[0]['word'] = " ".join(entities[0]['word'].split(' ')[1:])
                continuation = True
            elif token['end'] == entities[0]['end']:
                del entities[0]
                if not entities:
                    break
                continuation = False
    return tokens



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [17]:
def auto_NER_function(chunks, pdf_tokens):
    """
        INPUT: chunks - lista fragmentów dokumentu; pdf_tokens - tokeny całego PDF-a.
        OUTPUT: Tokeny z etykietami BIO, lista błędnych encji i lista encji o niskiej pewności.
        DESCRIPTION: Wykonuje pełny pipeline NER dla wszystkich chunków jednego PDF-a.
    """
    errors_entities = []
    low_confidence_entities = []
    i = 0
    for chunk in chunks:
        #print(i)
        entities = classify_tokens(chunk)
        if not entities:
            i=i+1
            continue
        entities = filter_entities(entities)
        if not entities:
            i=i+1
            continue
        entities, errors = validate_entities(entities, pdf_tokens)
        if not entities:
            i=i+1
            continue
        low_conf = flag_low_confidence(entities)
        pdf_tokens = mapping_entities_on_tokens(entities, pdf_tokens)
        errors_entities.extend(errors)
        low_confidence_entities.extend(low_conf)
        i=i+1

    return pdf_tokens, errors_entities, low_confidence_entities
        
        

In [18]:
pdf_tokens, errors, low_conf = auto_NER_function(chunks, pdf_tokens)[:5]
pdf_tokens[:5]

[{'token': 'Wyrok', 'start': 0, 'end': 5, 'label': 'O'},
 {'token': 'z', 'start': 6, 'end': 7, 'label': 'O'},
 {'token': 'dnia', 'start': 8, 'end': 12, 'label': 'O'},
 {'token': '20', 'start': 13, 'end': 15, 'label': 'O'},
 {'token': 'września', 'start': 16, 'end': 24, 'label': 'O'}]

In [19]:
import json


def load_json_data(path1, path2, path3):
    """
        INPUT: path1, path2, path3 - ścieżki do plików JSON ze słownikami.
        OUTPUT: Trzy listy: tytuły osób, dodatki/formy organizacji oraz miasta.
        DESCRIPTION: Wczytuje dane pomocnicze z plików JSON używane w postprocessingu encji.
    """
    with open(path1, encoding="utf-8") as f:
        person_titles = json.load(f)["PERSON_TITLES"]
    with open(path2, encoding="utf-8") as f:
        organization_titles = json.load(f)["ORG_ADDONS"]
    with open(path3, encoding="utf-8") as f:
        cities = json.load(f)["CITIES"]
    return person_titles, organization_titles, cities

def add_person_titles(tokens, person_titles):
    """
        INPUT: tokens - tokeny dokumentu; person_titles - lista tytułów osób.
        OUTPUT: Lista tokenów z poprawionymi etykietami PERSON.
        DESCRIPTION: Rozszerza encje PERSON o poprzedzające tytuły, np. dr, prof., sędzia.
    """
    titles = {title.lower().rstrip(".") for title in person_titles}
    for i, token in enumerate(tokens):
        if token.get("label") != "B-PERSON":
            continue
        
        title_indexes = []
        title_found = False
        non_title_before_name = 0
        j = i - 1

        while j >= 0:
            text = tokens[j]["token"]
            normalized = text.lower().rstrip(".")

            if text == ".":
                title_indexes.append(j)
                j -= 1
                continue

            if normalized in titles:
                title_indexes.append(j)
                title_found = True
                j -= 1
                continue

            if title_found:
                break

            non_title_before_name += 1

            if non_title_before_name == 2:
                break

            title_indexes.append(j)
            j -= 1

        if not title_found:
            continue

        title_indexes.reverse()

        tokens[title_indexes[0]]["label"] = "B-PERSON"

        for title_index in title_indexes[1:]:
            tokens[title_index]["label"] = "I-PERSON"

        tokens[i]["label"] = "I-PERSON"

    return tokens

In [20]:

PER_PATH = r"constants/PERSON_TITLES.json"
ORG_PATH = r"constants/ORG_ADDONS.json"
CITY_PATH = r"constants/CITIES.json"
PERSON_TITLES, ORG_TITLES, CITIES = load_json_data(PER_PATH, ORG_PATH, CITY_PATH)

In [24]:
pdf_tokens = add_person_titles(pdf_tokens, PERSON_TITLES)
pdf_tokens[19:43]

[{'token': '-', 'start': 86, 'end': 87, 'label': 'O'},
 {'token': 'Sądu', 'start': 87, 'end': 91, 'label': 'B-ORGANIZATION'},
 {'token': 'Ochrony', 'start': 92, 'end': 99, 'label': 'I-ORGANIZATION'},
 {'token': 'Konkurencji', 'start': 100, 'end': 111, 'label': 'I-ORGANIZATION'},
 {'token': 'i', 'start': 112, 'end': 113, 'label': 'I-ORGANIZATION'},
 {'token': 'Konsumentów', 'start': 114, 'end': 125, 'label': 'I-ORGANIZATION'},
 {'token': 'nie', 'start': 126, 'end': 129, 'label': 'O'},
 {'token': 'może', 'start': 130, 'end': 134, 'label': 'O'},
 {'token': 'sprowadzać', 'start': 135, 'end': 145, 'label': 'O'},
 {'token': 'się', 'start': 146, 'end': 149, 'label': 'O'},
 {'token': 'tylko', 'start': 150, 'end': 155, 'label': 'O'},
 {'token': 'do', 'start': 156, 'end': 158, 'label': 'O'},
 {'token': 'oceny', 'start': 159, 'end': 164, 'label': 'O'},
 {'token': 'legalności', 'start': 165, 'end': 175, 'label': 'O'},
 {'token': 'decyzji', 'start': 176, 'end': 183, 'label': 'O'},
 {'token': 'Pre',

In [25]:
def is_legal_form_token(token, legal_forms):
    """
        DESCRIPTION: Sprawdza, czy token odpowiada formie prawnej organizacji po normalizacji wielkości liter i kropki.
    """
    return token.lower().rstrip(".") in legal_forms


def is_legal_form_dot(tokens, index, legal_forms):
    """
        INPUT: tokens - lista tokenów; index - indeks sprawdzanej kropki; legal_forms - zbiór form prawnych.
        OUTPUT: True albo False.
        DESCRIPTION: Sprawdza, czy kropka jest częścią skrótu formy prawnej, np. S.A. albo sp. z o.o.
    """
    if tokens[index]["token"] != ".":
        return False

    prev_is_form = (
        index - 1 >= 0
        and is_legal_form_token(tokens[index - 1]["token"], legal_forms)
    )

    next_is_form = (
        index + 1 < len(tokens)
        and is_legal_form_token(tokens[index + 1]["token"], legal_forms)
    )

    return prev_is_form or next_is_form


def add_legal_forms_to_organization(pdf_tokens, legal_forms):
    """
        INPUT: pdf_tokens - tokeny dokumentu; legal_forms - lista form prawnych organizacji.
        OUTPUT: Lista tokenów z poprawionymi etykietami ORGANIZATION.
        DESCRIPTION: Rozszerza encje organizacji o sąsiadujące formy prawne przed lub po nazwie.
    """

    legal_forms = {
        form.lower().rstrip(".")
        for form in legal_forms
    }

    i = 0

    while i < len(pdf_tokens):
        if pdf_tokens[i].get("label") != "B-ORGANIZATION":
            i += 1
            continue

        org_start = i
        org_end = i

        while (
            org_end + 1 < len(pdf_tokens)
            and pdf_tokens[org_end + 1].get("label") == "I-ORGANIZATION"
        ):
            org_end += 1

        form_indexes = []
        j = org_start - 1

        while j >= 0:
            text = pdf_tokens[j]["token"]

            if is_legal_form_token(text, legal_forms):
                form_indexes.append(j)
                j -= 1
                continue

            if is_legal_form_dot(pdf_tokens, j, legal_forms):
                form_indexes.append(j)
                j -= 1
                continue

            break

        if form_indexes:
            form_indexes.reverse()

            pdf_tokens[form_indexes[0]]["label"] = "B-ORGANIZATION"

            for form_index in form_indexes[1:]:
                pdf_tokens[form_index]["label"] = "I-ORGANIZATION"

            pdf_tokens[org_start]["label"] = "I-ORGANIZATION"
            org_start = form_indexes[0]

        j = org_end + 1

        while j < len(pdf_tokens):
            text = pdf_tokens[j]["token"]

            if is_legal_form_token(text, legal_forms):
                pdf_tokens[j]["label"] = "I-ORGANIZATION"
                org_end = j
                j += 1
                continue

            if is_legal_form_dot(pdf_tokens, j, legal_forms):
                pdf_tokens[j]["label"] = "I-ORGANIZATION"
                org_end = j
                j += 1
                continue

            break

        i = org_end + 1

    return pdf_tokens

In [26]:
pdf_tokens = add_legal_forms_to_organization(pdf_tokens, ORG_TITLES)


In [27]:
def remove_invalid_city_tokens(pdf_tokens, CITIES):
    """
        INPUT: pdf_tokens - tokeny dokumentu; CITIES - lista poprawnych miast.
        OUTPUT: Lista tokenów z usuniętymi niepoprawnymi etykietami CITY.
        DESCRIPTION: Usuwa etykietę CITY z fragmentów, które nie występują na liście znanych miast.
    """
    cities = {city.lower() for city in CITIES}
    i = 0

    while i < len(pdf_tokens):
        if pdf_tokens[i].get("label") != "B-CITY":
            i += 1
            continue

        start = i
        end = i

        while (
            end + 1 < len(pdf_tokens)
            and pdf_tokens[end + 1].get("label") == "I-CITY"
        ):
            end += 1

        city_name = " ".join(
            token["token"] for token in pdf_tokens[start:end + 1]
        ).lower()

        if city_name not in cities:
            for j in range(start, end + 1):
                pdf_tokens[j]["label"] = "O"

        i = end + 1

    return pdf_tokens

def mark_city_tokens(pdf_tokens, CITIES):
    """
        INPUT: pdf_tokens - tokeny dokumentu; CITIES - lista miast.
        OUTPUT: Lista tokenów z oznaczonymi pojedynczymi tokenami miast.
        DESCRIPTION: Oznacza tokeny, których tekst pasuje do miasta ze słownika, etykietą B-CITY.
    """
    cities = {city.lower() for city in CITIES}

    for token in pdf_tokens:
        if token["token"].lower() in cities:
            token["label"] = "B-CITY"

    return pdf_tokens

In [28]:
pdf_tokens = remove_invalid_city_tokens(pdf_tokens, CITIES)
pdf_tokens = mark_city_tokens(pdf_tokens, CITIES)


In [29]:
def tokens_to_text(pdf_tokens):
    """
        INPUT: pdf_tokens - lista tokenów z pozycjami start i end.
        OUTPUT: Tekst odtworzony z tokenów.
        DESCRIPTION: Rekonstruuje tekst dokumentu z tokenów, zachowując odstępy wynikające z pozycji znakowych.
    """
    text_parts = []
    current_pos = 0

    for token in pdf_tokens:
        start = token["start"]
        end = token["end"]

        if start > current_pos:
            text_parts.append(" " * (start - current_pos))

        text_parts.append(token["token"])
        current_pos = end

    return "".join(text_parts)


def extract_entities(pdf_tokens):
    """
        INPUT: pdf_tokens - tokeny dokumentu z etykietami BIO.
        OUTPUT: Lista encji z polami text, label, start i end.
        DESCRIPTION: Łączy kolejne tokeny B-/I- w pełne encje do zapisu w formacie JSONL.
    """
    entities = []
    current_entity = None

    for token in pdf_tokens:
        label = token.get("label", "O")

        if label == "O":
            if current_entity is not None:
                entities.append(current_entity)
                current_entity = None
            continue

        prefix, entity_label = label.split("-", 1)

        if prefix == "B":
            if current_entity is not None:
                entities.append(current_entity)

            current_entity = {
                "text": token["token"],
                "label": entity_label,
                "start": token["start"],
                "end": token["end"],
            }

        elif prefix == "I" and current_entity is not None:
            current_entity["text"] += " " + token["token"]
            current_entity["end"] = token["end"]

    if current_entity is not None:
        entities.append(current_entity)

    return entities


def save_pdfs_tokens_to_jsonl(pdf_tokens, output_path="output.jsonl"):
    """
        INPUT: pdf_tokens - tokeny jednego PDF-a; output_path - ścieżka do pliku JSONL.
        OUTPUT: Brak zwracanej wartości; zapisuje rekord do pliku.
        DESCRIPTION: Zapisuje jeden dokument do JSONL, nadpisując wcześniejszą zawartość pliku.
    """
    with open(output_path, "w", encoding="utf-8") as f:
        record = {
            "text": tokens_to_text(pdf_tokens),
            "tokens": [token["token"] for token in pdf_tokens],
            "labels": [token.get("label", "O") for token in pdf_tokens],
            "entities": extract_entities(pdf_tokens),
        }

        f.write(json.dumps(record, ensure_ascii=False) + "\n")

In [25]:
#save_pdfs_tokens_to_jsonl(pdf_tokens)

In [ ]:
def append_pdf_tokens_to_jsonl(pdf_tokens, output_path="output.jsonl"):
    """
        INPUT: pdf_tokens - tokeny jednego PDF-a; output_path - ścieżka do pliku JSONL.
        OUTPUT: Brak zwracanej wartości; dopisuje rekord do pliku.
        DESCRIPTION: Dodaje jeden dokument jako nową linię JSONL bez usuwania wcześniejszych wyników.
    """
    record = {
        "text": tokens_to_text(pdf_tokens),
        "tokens": [token["token"] for token in pdf_tokens],
        "labels": [token.get("label", "O") for token in pdf_tokens],
        "entities": extract_entities(pdf_tokens),
    }

    with open(output_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


OUTPUT_PATH = "output.jsonl"

BATCH_START = 105
BATCH_SIZE = 3
CLEAR_OUTPUT = False  # tylko przy pierwszej paczce True

if CLEAR_OUTPUT:
    open(OUTPUT_PATH, "w", encoding="utf-8").close()

batch_pdf_paths = pdf_paths[BATCH_START:BATCH_START + BATCH_SIZE]

errors_by_pdf = []

for i, pdf_path in enumerate(batch_pdf_paths, start=BATCH_START + 1):
    try:
        print(f"[{i}/{len(pdf_paths)}] Przetwarzam: {pdf_path}")

        pdf_text = extract_text_from_pdf(pdf_path)
        pdf_tokens = tokenize_text(pdf_text)
        chunks = create_chunks(text=pdf_text, tokens=pdf_tokens)

        pdf_tokens, errors, low_conf = auto_NER_function(chunks, pdf_tokens)

        pdf_tokens = add_person_titles(pdf_tokens, PERSON_TITLES)
        pdf_tokens = add_legal_forms_to_organization(pdf_tokens, ORG_TITLES)
        pdf_tokens = remove_invalid_city_tokens(pdf_tokens, CITIES)
        pdf_tokens = mark_city_tokens(pdf_tokens, CITIES)

        append_pdf_tokens_to_jsonl(pdf_tokens, OUTPUT_PATH)

        print(f"OK: zapisano {len(pdf_tokens)} tokenów\n")

    except Exception as e:
        errors_by_pdf.append({
            "pdf_path": pdf_path,
            "error": str(e),
        })
        print(f"BŁĄD przy PDF-ie: {pdf_path}")
        print(f"{type(e).__name__}: {e}\n")
        continue

[106/108] Przetwarzam: data\zbior_danych\rekrutacja.pk.edu.pl\2026-01\TCh_kryteria-kwalifikacyjne-rekrutacji-na-II-stopien_9469a253.pdf
Text extraction from: TCh_kryteria-kwalifikacyjne-rekrutacji-na-II-stopien_9469a253.pdf

Extracted text from TCh_kryteria-kwalifikacyjne-rekrutacji-na-II-stopien_9469a253.pdf: 48059 characters

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
OK: zapisano 7717 tokenów

[107/108] Przetwarzam: data\zbior_danych\rekrutacja.pk.edu.pl\2026-01\technik-2025-26_1ca3d546.pdf
Text extraction from: technik-2025-26_1ca3d546.pdf

Extracted text from technik-2025-26_1ca3d546.pdf: 0 characters

OK: zapisano 0 tokenów

[108/108] Przetwarzam: data\zbior_danych\rekrutacja.pk.edu.pl\2026-01\WA_ST2_Portfolio_warunki_technczne_tryb_przekazywania_91f64c74.pdf
Text extraction from: WA_ST2_Portfolio_warunki_technczne_tryb_przekazywania_91f64c74.pdf

Extracted text from WA_ST2_Port